In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
pd.set_option("display.max_columns", None)
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from utils.modelling import *
from utils.plotting import *

In [ ]:
import os
if os.path.exists("../data/claims_evaluation.parquet"):
    evaluation_df = pd.read_parquet("../data/claims_evaluation.parquet")
else:
    evaluation_df = pd.DataFrame()

## VERSION 1

In [ ]:
insurance_claims = pd.read_parquet("../data/claims_model_dataset_v1.parquet")

numeric_columns = insurance_claims.select_dtypes(include = ["int64", "int32", "float64"]).columns.drop("total_claim_amount")
categorical_columns = insurance_claims.select_dtypes(include = ["object"]).columns

insurance_claims = encode_features(insurance_claims, categorical_columns)
x_train, x_test, y_train, y_test = split_dataset(insurance_claims, "total_claim_amount", 0.2)

preprocessor = ColumnTransformer(
    transformers = [
        ("numeric_scaled", StandardScaler(), numeric_columns)
    ], 
    remainder = "passthrough"
)

#### LINEAR REGRESSION

In [ ]:
pipeline = create_pipeline(LinearRegression(), preprocessor)
lr_v1 = transform_target(pipeline, np.log1p, np.expm1)
lr_v1.fit(x_train, y_train)
lr_v1_results = regression_results(lr_v1, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V1", "Linear Regression", "Initial", lr_v1_results)

#### RANDOM FOREST

In [ ]:
pipeline = create_pipeline(RandomForestRegressor(random_state = 123), preprocessor)
rfr_v1 = transform_target(pipeline, np.log1p, np.expm1)
rfr_v1.fit(x_train, y_train)
rfr_v1_results = regression_results(rfr_v1, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V1", "Random Forest", "Initial", rfr_v1_results)

In [ ]:
parameters = {
    "regressor__model__n_estimators": [300], 
    "regressor__model__min_samples_split": [3],
    "regressor__model__min_samples_leaf": [4], 
    "regressor__model__max_depth": [3],
    "regressor__model__max_features": [0.75]
}

rfr_v1_gs = optimise_model(rfr_v1, parameters, "neg_root_mean_squared_error", 5, x_train, y_train)
rfr_v1_gs_results = regression_results(rfr_v1_gs, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V1", "Random Forest", "Optimised", rfr_v1_gs_results)

#### GRADIENT BOOSTING

In [ ]:
pipeline = create_pipeline(GradientBoostingRegressor(random_state = 123), preprocessor)
gbr_v1 = transform_target(pipeline, np.log1p, np.expm1)
gbr_v1.fit(x_train, y_train)
gbr_v1_results = regression_results(gbr_v1, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V1", "Gradient Boosting", "Initial", gbr_v1_results)

In [ ]:
parameters = {
    'regressor__model__learning_rate': [0.01],
    'regressor__model__n_estimators': [350],
    'regressor__model__max_depth': [3],
    'regressor__model__min_samples_split': [3],
    'regressor__model__min_samples_leaf': [3]
}

gbr_v1_gs = optimise_model(gbr_v1, parameters, "neg_root_mean_squared_error", 5, x_train, y_train)
gbr_v1_gs_results = regression_results(gbr_v1_gs, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V1", 
#                                         "Gradient Boosting", "Optimised", gbr_v1_gs_results)

### COMPARISON

In [ ]:
v1_comparison = evaluation_df[evaluation_df["dataset_version"] == "V1"]
v1_comparison.loc[[0, 2, 4]]

In [ ]:
dict_for_df = {
    "Linear Regression": lr_v1.regressor_.named_steps["model"].coef_,
    "Random Forest": rfr_v1_gs.regressor_.named_steps["model"].feature_importances_,
    "Gradient Boosting": gbr_v1_gs.regressor_.named_steps["model"].feature_importances_
}

plot_feature_importance(x_train, dict_for_df, 0, 10)

## VERSION 2

In [ ]:
insurance_claims = pd.read_parquet("../data/claims_model_dataset_v2.parquet")

numeric_columns = insurance_claims.select_dtypes(include = ["int64", "int32", "float64"]).columns.drop("total_claim_amount")
categorical_columns = insurance_claims.select_dtypes(include = ["object"]).columns

insurance_claims = encode_features(insurance_claims, categorical_columns)
x_train, x_test, y_train, y_test = split_dataset(insurance_claims, "total_claim_amount", 0.2)

preprocessor = ColumnTransformer(
    transformers = [
        ("numeric_scaled", StandardScaler(), numeric_columns)
    ], 
    remainder = "passthrough"
)

#### LINEAR REGRESSION

In [ ]:
pipeline = create_pipeline(LinearRegression(), preprocessor)
lr_v2 = transform_target(pipeline, np.log1p, np.expm1)
lr_v2.fit(x_train, y_train)
lr_v2_results = regression_results(lr_v2, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V2", "Linear Regression", "Initial", lr_v2_results)

#### RANDOM FOREST

In [ ]:
pipeline = create_pipeline(RandomForestRegressor(random_state = 123), preprocessor)
rfr_v2 = transform_target(pipeline, np.log1p, np.expm1)
rfr_v2.fit(x_train, y_train)
rfr_v2_results = regression_results(rfr_v2, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V2", "Random Forest", "Initial", rfr_v2_results)

In [ ]:
parameters = {
    "regressor__model__n_estimators": [325], 
    "regressor__model__min_samples_split": [3],
    "regressor__model__min_samples_leaf": [5], 
    "regressor__model__max_depth": [3],
    "regressor__model__max_features": [0.65]
}

rfr_v2_gs = optimise_model(rfr_v2, parameters, "neg_root_mean_squared_error", 5, x_train, y_train)
rfr_v2_gs_results = regression_results(rfr_v2_gs, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V2", "Random Forest", "Optimised", rfr_v2_gs_results)

#### GRADIENT BOOSTING

In [ ]:
pipeline = create_pipeline(GradientBoostingRegressor(random_state = 123), preprocessor)
gbr_v2 = transform_target(pipeline, np.log1p, np.expm1)
gbr_v2.fit(x_train, y_train)
gbr_v2_results = regression_results(gbr_v2, x_train, x_test, y_train, y_test)

evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V2", "Gradient Boosting", "Initial", gbr_v2_results)

In [ ]:
parameters = {
    'regressor__model__learning_rate': [0.01],
    'regressor__model__n_estimators': [375],
    'regressor__model__max_depth': [2],
    'regressor__model__min_samples_split': [2],
    'regressor__model__min_samples_leaf': [6]
}

gbr_v2_gs = optimise_model(gbr_v2, parameters, "neg_root_mean_squared_error", 5, x_train, y_train)
gbr_v2_gs_results = regression_results(gbr_v2_gs, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V2", 
#                                         "Gradient Boosting", "Optimised", gbr_v2_gs_results)

### COMPARISON

In [ ]:
v2_comparison = evaluation_df[evaluation_df["dataset_version"] == "V2"]
v2_comparison.loc[[5, 7, 9]]

In [ ]:
dict_for_df = {
    "Linear Regression": lr_v2.regressor_.named_steps["model"].coef_,
    "Random Forest": rfr_v2_gs.regressor_.named_steps["model"].feature_importances_,
    "Gradient Boosting": gbr_v2_gs.regressor_.named_steps["model"].feature_importances_
}

plot_feature_importance(x_train, dict_for_df, 0, 10)

In [ ]:
plot_feature_importance(x_train, dict_for_df, 20, 0, True)

## VERSION 3

In [ ]:
insurance_claims = pd.read_parquet("../data/claims_model_dataset_v3.parquet")

numeric_columns = insurance_claims.select_dtypes(include = ["int64", "int32", "float64"]).columns.drop("total_claim_amount")
categorical_columns = insurance_claims.select_dtypes(include = ["object"]).columns

insurance_claims = encode_features(insurance_claims, categorical_columns)
x_train, x_test, y_train, y_test = split_dataset(insurance_claims, "total_claim_amount", 0.2)

preprocessor = ColumnTransformer(
    transformers = [
        ("numeric_scaled", StandardScaler(), numeric_columns)
    ], 
    remainder = "passthrough"
)

#### LINEAR REGRESSION

In [ ]:
pipeline = create_pipeline(LinearRegression(), preprocessor)
lr_v3 = transform_target(pipeline, np.log1p, np.expm1)
lr_v3.fit(x_train, y_train)
lr_v3_results = regression_results(lr_v3, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V3", "Linear Regression", "Initial", lr_v3_results)

#### RANDOM FOREST

In [ ]:
pipeline = create_pipeline(RandomForestRegressor(random_state = 123), preprocessor)
rfr_v3 = transform_target(pipeline, np.log1p, np.expm1)
rfr_v3.fit(x_train, y_train)
rfr_v3_results = regression_results(rfr_v3, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V3", "Random Forest", "Initial", rfr_v3_results)

In [ ]:
parameters = {
    "regressor__model__n_estimators": [300], 
    "regressor__model__min_samples_split": [2],
    "regressor__model__min_samples_leaf": [10], 
    "regressor__model__max_depth": [2],
    "regressor__model__max_features": [0.9]
}

rfr_v3_gs = optimise_model(rfr_v3, parameters, "neg_root_mean_squared_error", 5, x_train, y_train)
rfr_v3_gs_results = regression_results(rfr_v3_gs, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V3", "Random Forest", "Optimised", rfr_v3_gs_results)

#### GRADIENT BOOSTING

In [ ]:
pipeline = create_pipeline(GradientBoostingRegressor(random_state = 123), preprocessor)
gbr_v3 = transform_target(pipeline, np.log1p, np.expm1)
gbr_v3.fit(x_train, y_train)
gbr_v3_results = regression_results(gbr_v3, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V3", "Gradient Boosting", "Initial", gbr_v3_results)

In [ ]:
parameters = {
    'regressor__model__learning_rate': [0.025],
    'regressor__model__n_estimators': [175],
    'regressor__model__max_depth': [2],
    'regressor__model__min_samples_split': [2],
    'regressor__model__min_samples_leaf': [5]
}

gbr_v3_gs = optimise_model(gbr_v3, parameters, "neg_root_mean_squared_error", 5, x_train, y_train)
gbr_v3_gs_results = regression_results(gbr_v3_gs, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V3", 
#                                         "Gradient Boosting", "Optimised", gbr_v3_gs_results)

### COMPARISON

In [ ]:
v3_comparison = evaluation_df[evaluation_df["dataset_version"] == "V3"]
v3_comparison.loc[[10, 12, 14]]

In [ ]:
dict_for_df = {
    "Linear Regression": lr_v3.regressor_.named_steps["model"].coef_,
    "Random Forest": rfr_v3_gs.regressor_.named_steps["model"].feature_importances_,
    "Gradient Boosting": gbr_v3_gs.regressor_.named_steps["model"].feature_importances_
}

plot_feature_importance(x_train, dict_for_df, 0, 10)

In [ ]:
best_model_features = {
    "Column": x_train.columns,
    "Feature Importance": rfr_v3_gs.regressor_.named_steps["model"].feature_importances_
}

best_model_features_df = pd.DataFrame.from_dict(best_model_features)
best_model_features_df.to_parquet("../data/best_regression_model_features.parquet")